# GWB (log10_A = -14.5, γ = 13/3) su scattering α = 4.4, analizzato con α LIBERO

Dati: `sim_data_noiseGWB_A1405_g433_sv44/` — red noise + DM + scattering generato con α = 4.4 + GWB con γ = 13/3 e log10_A = -14.5, seed `jax.random.key(1)`.

Analisi: 7 parametri, scattering con `ds.dmfourierbasis_alpha` (α libero), **nessun termine GWB**. Il GWB non modellato e l'indice cromatico ignoto agiscono insieme: è la configurazione più vicina al caso reale.

Il campionamento si lancia **da qui**, come negli altri notebook: le celle di prior, likelihood, sampler e `sample(...)` sono più sotto. La catena finisce in `./test_gwb_A1405_sv44_alphafree/`.

In alternativa, lo stesso campionamento si può lanciare da terminale, senza occupare il kernel:

```bash
caffeinate -i python run_08_alpha_libero.py --data sim_data_noiseGWB_A1405_g433_sv44/J1640+2224 --outdir ./test_gwb_A1405_sv44_alphafree/ 2>&1 | tee run_gwb1405.log
```

Le due strade scrivono lo stesso `chain_1.txt`, quindi va usata una sola delle due per volta.

Le celle dei grafici leggono la catena da disco e funzionano anche su una catena **parziale**: stampano quanti campioni ci sono e l'ESS, e lo riportano nei titoli delle figure.

## Setup comune

Copia delle celle 0, 1, 3, 4, 6, 7, 9 di `Likelihood.ipynb`: serve per `logL.params`, `my_priordict` e `build_priors`.

In [ ]:
import numpy as np
import os, glob, json
import pickle

import sys

# this is the path to the discovery folder
sys.path.insert(0, "./")
import discovery_new as ds

import matplotlib.pyplot as plt
import scipy
import jax.numpy as jnp

from scipy.stats import uniform
import re
from PTMCMCSampler.PTMCMCSampler import PTSampler as ptmcmc
from scipy.stats import uniform
import jax

import time
import emcee

In [ ]:
# path to the folder with the data
path = './dati/DR2low_feathers/'
s_dsfiles = np.sort(os.listdir(path))
psr = [ds.pulsar.Pulsar.read_feather(path + f'{psrfile}') for psrfile in s_dsfiles][9] # we are only analysing pulsar #9

In [ ]:
# data
model = [psr.residuals]
# timing model
model += [ds.makegp_timing(psr, svd=True)]
# white noise parameters
model += [ds.makenoise_measurement(psr, psr.noisedict, tnequad = True, ecorr=True)]
# red noise paramters
model.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_red_components'], T=ds.getspan(psr), name='red_noise' ))
# DM variation parameters
model.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_dm_components'], T=ds.getspan(psr), name='dm_gp', fourierbasis=ds.make_dmfourierbasis(alpha = 2.0, tndm = False)))
# scattering noise parameters
model.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_chrom_components'], T=ds.getspan(psr), name='sv_gp', fourierbasis=ds.make_dmfourierbasis(alpha = 4.0, tndm = False)))

In [ ]:
logL = ds.PulsarLikelihood(model).logL

In [ ]:
# annoying function needed because the function logL want a dictionary as input, but our sampler wants a likelihood with an array as input
# so we have to switch from one to another :(
def params_dict_from_array(x):
    # ensure x is a plain numpy array before slicing
    x = np.asarray(x)
    return dict(zip(logL.params, x))

In [ ]:
##### Cell for the prior distributions ###################################################################################

# dictionary with minimum and maximum value of the prior distribution
my_priordict = {}
my_priordict["(.*_)?sv_gp_gamma.*"] = [0, 7]
my_priordict["(.*_)?sv_gp_alpha.*"] = [0, 10]
my_priordict["(.*_)?sv_gp_log10_A.*"] = [-20, -11]
my_priordict["(.*_)?sw_log10_A.*"] = [-10, 1]
my_priordict["(.*_)?sw_gamma.*"] = [-5, 6]
my_priordict["(.*_)?dm_gp_gamma.*"] = [0, 7]
my_priordict["(.*_)?dm_gp_log10_A.*"] = [-20, -11]
my_priordict["(.*_)?red_noise_gamma.*"] = [0, 7]
my_priordict["(.*_)?red_noise_log10_A.*"] = [-20, -11]
my_priordict["(.*_)?sw_n_earth.*"] = [0, 30]


def build_priors(param_names, priordict):
    """
    Build a {param_name: scipy dist} dict with uniform prior distributions in the range
    """
    priors = {}
    unmatched = []
    for name in param_names:
        matched = False
        for pattern, (lo, hi) in priordict.items():
            if re.fullmatch(pattern, name):
                priors[name] = uniform(loc=lo, scale=hi - lo) ## this is where the uniform distribution is defined
                matched = True
                break
        if not matched:
            unmatched.append(name)

    if unmatched:
        print(f"Warning: no prior found for {len(unmatched)} parameters, "
              f"falling back to standard normal:\n  {unmatched}")
    return priors

priors = build_priors(logL.params, my_priordict)
priors

def log_prior(x):
    """Compute log prior"""
    params = params_dict_from_array(x)
    
    logp = 0.0
    for name, value in params.items():
        if name in priors:
            logp += priors[name].logpdf(value)
        else:
            # Default: standard normal prior
            if isinstance(value, np.ndarray):
                logp += -0.5 * np.sum(value**2)
            else:
                logp += -0.5 * value**2
    
    return logp

def prior_draw():
    """Draw from prior and return as flat array"""
    x = []
    for name in logL.params:
        x.append(priors[name].rvs())
    return np.array(x)

x0 = prior_draw()
x0

In [ ]:
@jax.jit # compile the likelihood in jax to make it faster
def _jit_logL(params_array):
    params = dict(zip(logL.params, params_array))
    return logL(params)

_jit_logL(x0)

def log_likelihood(x): 
    """
    Likelihood function that is given to the samples, it uses the jitted (=faster) version of the likelihood
    """
    x = np.asarray(x, dtype=np.float64)
    result = _jit_logL(jnp.array(x))
    return float(np.array(result))  # always a plain Python float

## Dati e modello con α libero

In [ ]:
gwb1405_psr = ds.pulsar.Pulsar.read_feather('sim_data_noiseGWB_A1405_g433_sv44/J1640+2224')
print('nome       :', gwb1405_psr.name)
print('N_TOA      :', len(gwb1405_psr.toas))
print('std residui:', np.std(gwb1405_psr.residuals))

# contributo del GWB: differenza rispetto alla simulazione senza GWB (stesso seed, stesso alpha)
sv44_psr = ds.pulsar.Pulsar.read_feather('sim_data_noise_sv44/J1640+2224')
print('std del solo GWB:', np.std(gwb1405_psr.residuals - sv44_psr.residuals))

In [ ]:
model_gwb1405 = [gwb1405_psr.residuals]
model_gwb1405 += [ds.makegp_timing(gwb1405_psr, svd=True)]
model_gwb1405 += [ds.makenoise_measurement(gwb1405_psr, gwb1405_psr.noisedict, tnequad=True, ecorr=True)]
model_gwb1405.append(ds.makegp_fourier(gwb1405_psr, ds.powerlaw, gwb1405_psr.noisedict[gwb1405_psr.name + '_red_components'],
                                   T=ds.getspan(gwb1405_psr), name='red_noise'))
model_gwb1405.append(ds.makegp_fourier(gwb1405_psr, ds.powerlaw, gwb1405_psr.noisedict[gwb1405_psr.name + '_dm_components'],
                                   T=ds.getspan(gwb1405_psr), name='dm_gp',
                                   fourierbasis=ds.make_dmfourierbasis(alpha=2.0, tndm=False)))
model_gwb1405.append(ds.makegp_fourier(gwb1405_psr, ds.powerlaw, gwb1405_psr.noisedict[gwb1405_psr.name + '_chrom_components'],
                                   T=ds.getspan(gwb1405_psr), name='sv_gp',
                                   fourierbasis=ds.dmfourierbasis_alpha))   # <-- alpha libero

logL_gwb1405 = ds.PulsarLikelihood(model_gwb1405).logL
logL_gwb1405.params   # 7 parametri, sv_gp_alpha compreso

In [ ]:
atteso = set(logL.params) | {'J1640+2224_sv_gp_alpha'}
assert set(logL_gwb1405.params) == atteso, \
    f"Mancanti: {atteso - set(logL_gwb1405.params)}\nIn eccesso: {set(logL_gwb1405.params) - atteso}"
print('OK: 7 parametri attesi')
idx = {name: i for i, name in enumerate(logL_gwb1405.params)}
idx   # ordine delle colonne della catena

## Prior, likelihood e sampler

Stessa struttura del notebook 08. `sv_gp_alpha` prende la prior uniforme [0, 10] già presente in `my_priordict`.

In [ ]:
def params_dict_from_array_gwb1405(x):
    x = np.asarray(x)
    return dict(zip(logL_gwb1405.params, x))

priors_gwb1405 = build_priors(logL_gwb1405.params, my_priordict)
print('prior su alpha:', priors_gwb1405['J1640+2224_sv_gp_alpha'].support())

def log_prior_gwb1405(x):
    params = params_dict_from_array_gwb1405(x)
    logp = 0.0
    for name, value in params.items():
        if name in priors_gwb1405:
            logp += priors_gwb1405[name].logpdf(value)
        else:
            logp += -0.5 * np.sum(np.asarray(value)**2)
    return logp

def prior_draw_gwb1405():
    return np.array([priors_gwb1405[name].rvs() for name in logL_gwb1405.params])

x0_gwb1405 = prior_draw_gwb1405()
print('x0 =', x0_gwb1405)

In [ ]:
@jax.jit
def _jit_logL_gwb1405(params_array):
    return logL_gwb1405(dict(zip(logL_gwb1405.params, params_array)))

def log_likelihood_gwb1405(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.array(_jit_logL_gwb1405(jnp.array(x))))

print('logL iniziale =', log_likelihood_gwb1405(x0_gwb1405))

# tempo di una valutazione, per stimare la durata
start = time.time()
for _ in range(20):
    log_likelihood_gwb1405(x0_gwb1405)
dt = (time.time() - start) / 20
print(f'una valutazione = {dt*1e3:.1f} ms  ->  10^6 iterazioni ~ {dt*1e6/3600:.1f} ore (solo likelihood)')

In [ ]:
sampler_gwb1405 = ptmcmc(
    ndim=len(logL_gwb1405.params),
    logl=log_likelihood_gwb1405,
    logp=log_prior_gwb1405,
    cov=np.eye(len(logL_gwb1405.params)) * 0.001,
    outDir='./test_gwb_A1405_sv44_alphafree/',
    resume=True
)

### Sampling

Con α libero una valutazione costa circa 50 ms sul Mac, quindi 10⁶ iterazioni sono circa 14 ore. Il kernel resta occupato per tutta la durata: niente altre celle fino alla fine.

Per interromperla: **Kernel → Interrupt**. Con `resume=True` la catena riparte da dove era, quindi puoi fermarla, fare i grafici su quella parziale e riprendere dopo.

Equivalente da terminale, che non occupa Jupyter e sopravvive alla chiusura del browser:

```bash
caffeinate -i python run_08_alpha_libero.py --data sim_data_noiseGWB_A1405_g433_sv44/J1640+2224 --outdir ./test_gwb_A1405_sv44_alphafree/ 2>&1 | tee run_gwb1405.log
```

Le due strade scrivono lo stesso `chain_1.txt`: scegli quella che preferisci, **ma non lanciarle insieme sulla stessa cartella**.

In [ ]:
print("Starting sampling (GWB log10_A=-14.5, gamma=13/3 su scattering alpha=4.4, analisi con alpha LIBERO)...")
sampler_gwb1405.sample(x0_gwb1405, Niter=1000000, thin=10)

## Catena

Prodotta dallo script. Se stai leggendo una catena ancora in corso, il numero di campioni è minore di 75.000 e i grafici vanno letti come indicativi.

In [ ]:
chain_gwb1405 = np.loadtxt('./test_gwb_A1405_sv44_alphafree/chain_1.txt')
n_tot = len(chain_gwb1405)
chain_gwb1405 = chain_gwb1405[n_tot//4:, :-4]          # 25% di burn-in, via le 4 colonne diagnostiche
print(f'righe nel file: {n_tot}  ->  campioni usati: {len(chain_gwb1405)}  ({100*n_tot/100000:.1f}% della run completa)')

import emcee
tau_gwb1405 = emcee.autocorr.integrated_time(chain_gwb1405, has_walkers=False, quiet=True)
ess_gwb1405 = int(len(chain_gwb1405) / tau_gwb1405.max())
print('tau  =', dict(zip([p.replace('J1640+2224_','') for p in logL_gwb1405.params], np.round(tau_gwb1405, 1))))
print('ESS  =', ess_gwb1405, '(sotto ~200 i valori numerici non sono affidabili, la forma della posterior si')
nota = f'catena parziale: {len(chain_gwb1405)} campioni, ESS ~ {ess_gwb1405}'

In [ ]:
# valori veri dei processi di rumore (alpha = 4.4 per costruzione; il GWB non e' nel modello)
truths_dict = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
    'J1640+2224_sv_gp_alpha': 4.4,
}
truth_values = [truths_dict[p] for p in logL_gwb1405.params]
clean_labels = [p.replace('J1640+2224_', '') for p in logL_gwb1405.params]

print(f"{'parametro':20s} {'vero':>9s} | {'mediana [16, 84]':>30s}")
for name in logL_gwb1405.params:
    q16, q50, q84 = np.percentile(chain_gwb1405[:, idx[name]], [16, 50, 84])
    print(f"{name.replace('J1640+2224_', ''):20s} {truths_dict[name]:9.3f} | {q50:9.3f} [{q16:9.3f}, {q84:9.3f}]")

### Corner a 7 parametri

In [ ]:
import corner

fig = corner.corner(
    chain_gwb1405,
    labels=clean_labels,
    truths=truth_values,
    truth_color='#222222',
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
    color='#A06CD5',
)
fig.set_size_inches(18, 18)
fig.suptitle(f'J1640+2224 — GWB log10_A=-14.5, γ=13/3 su scattering α=4.4, analisi con α libero e senza GWB\n{nota}',
             fontsize=14, y=1.00)
fig.savefig('corner_gwb_A1405_sv44_alphafree.png', dpi=150, bbox_inches='tight')

### Degenerazione α – log10_A dello scattering

La retta rossa è la direzione prevista, di pendenza −log₁₀(1400/ν_eff), con ν_eff pesata sul contributo dei TOA al segnale di scattering. Passa per il punto vero.

In [ ]:
f_toa = np.asarray(gwb1405_psr.freqs)
w = (1400.0 / f_toa)**(2 * 4.4) / np.asarray(gwb1405_psr.toaerrs)**2
slope = -np.sum(w * np.log10(1400.0 / f_toa)) / np.sum(w)
print(f'nu_eff ~ {1400.0 / 10**(-slope):.0f} MHz, pendenza attesa = {slope:.2f}')

a_s  = chain_gwb1405[:, idx['J1640+2224_sv_gp_alpha']]
lA_s = chain_gwb1405[:, idx['J1640+2224_sv_gp_log10_A']]
print('correlazione di Pearson alpha - log10_A =', np.corrcoef(a_s, lA_s)[0, 1])

fig = corner.corner(
    np.column_stack([a_s, lA_s]),
    labels=['sv_gp_alpha', 'sv_gp_log10_A'],
    truths=[4.4, truths_dict['J1640+2224_sv_gp_log10_A']],
    truth_color='#222222',
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
    color='#A06CD5',
)
ax = fig.axes[2]
xx = np.linspace(a_s.min(), a_s.max(), 100)
ax.plot(xx, truths_dict['J1640+2224_sv_gp_log10_A'] + slope * (xx - 4.4), 'r--', lw=1.5,
        label=f'degenerazione attesa (pendenza {slope:.2f})')
ax.legend(fontsize=9)
fig.set_size_inches(8, 8)
fig.suptitle(nota, fontsize=10)
fig.savefig('corner_gwb_A1405_sv44_alphafree_degenerazione.png', dpi=150, bbox_inches='tight')

### Effetto del GWB: confronto con la baseline senza GWB

`test_sim_sv44_alphafree/` è la stessa analisi con α libero sugli stessi processi di rumore ma **senza** GWB. Stesso seed, stesso modello: la differenza è dovuta al solo GWB. La cella salta il confronto se quella catena non c'è ancora.

In [ ]:
import os

base_path = './test_sim_sv44_alphafree/chain_1.txt'
if not os.path.exists(base_path):
    print('baseline assente, confronto saltato:', base_path)
else:
    base = np.loadtxt(base_path)
    nb_tot = len(base)
    base = base[nb_tot//4:, :-4]
    print(f'baseline: {len(base)} campioni ({100*nb_tot/100000:.1f}% della run completa)')

    print(f"\n{'parametro':20s} {'vero':>9s} | {'senza GWB':>20s} | {'con GWB':>20s}")
    for name in logL_gwb1405.params:
        a50 = np.median(base[:, idx[name]])
        b50 = np.median(chain_gwb1405[:, idx[name]])
        print(f"{name.replace('J1640+2224_', ''):20s} {truths_dict[name]:9.3f} | {a50:20.3f} | {b50:20.3f}")

    fig = corner.corner(
        base,
        labels=clean_labels,
        truths=truth_values,
        truth_color='#222222',
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
        color='#457B9D',
    )
    corner.corner(
        chain_gwb1405,
        labels=clean_labels,
        truths=truth_values,
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
        color='#A06CD5',
        fig=fig
    )
    fig.set_size_inches(18, 18)
    fig.suptitle('J1640+2224 — α libero: senza GWB (blu) vs con GWB log10_A=-14.5, γ=13/3 (colore)\ncatene parziali',
                 fontsize=14, y=1.00)
    fig.savefig('corner_alphafree_senza_vs_gwb_A1405.png', dpi=150, bbox_inches='tight')

## Riepilogo

_Da compilare._

- α recuperato: …
- `sv_gp_log10_A` rispetto al valore vero: …
- Effetto del GWB sui parametri di rumore (soprattutto red noise): …
- Stato della catena al momento dei grafici: …